### Import Dependencies

In [ ]:
from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langsmith import traceable

import instructor

from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt


from langchain_core.messages import SystemMessage, convert_to_openai_messages, HumanMessage, AIMessage
from IPython.display import Image, display

from typing import Any, Annotated, List, Literal
from pydantic import Field
from operator import add

from jinja2 import Template

from utils.utils import postprocess_response
from utils.tools import get_formatted_item_context, get_formatted_reviews_context, get_shopping_cart, remove_from_cart, add_to_shopping_cart, check_warehouse_availability, reserve_warehouse_items

from langchain_litellm import ChatLiteLLM

from dotenv import load_dotenv

load_dotenv("../../.env")

### LangChain LiteLLM Introduction

In [ ]:
llm = ChatLiteLLM(
    model="openai/gpt-5.4-mini",
    reasoning_effort="low",
    use_responses_api=True
)

response = llm.invoke(
    [
        SystemMessage(content="What kind of model family are you?")
    ]
)

In [ ]:
response

In [ ]:
print(response.content)

In [ ]:
llm = ChatLiteLLM(
    model="groq/llama-3.3-70b-versatile"
)

response = llm.invoke(
    [
        SystemMessage(content="What kind of model family are you?")
    ]
)

In [ ]:
print(response.content)

## Coordinator Agent

In [ ]:
class AgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    messages: Annotated[List[Any], add_messages] = []

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    next_agent: str = ""
    next_agent_task: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    user_intent: str = ""
    product_qna_agent: AgentProperties = AgentProperties()
    shopping_cart_agent: AgentProperties = AgentProperties()
    warehouse_manager_agent: AgentProperties = AgentProperties()
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""
    user_id: str = ""
    cart_id: str = ""

In [ ]:
class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

In [ ]:
class Plan(BaseModel):
    
    next_agent: str = Field(description="The next agent to invoke")
    next_agent_task: str = Field(description="The task to be performed by the next agent")

In [ ]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available agents and skills
 
### `product_qna_agent`
 
Owns all product knowledge: catalog, specs, pricing, reviews, recommendations. This is the default agent for any question about what exists or what to buy.
 
Skills:
 
- `get_formatted_item_context` — Search available products and return the top k matching inventory items.
- `get_formatted_reviews_context` — Get the top k reviews matching a query for a list of prefiltered items.
Does not: modify the cart, check warehouse stock, or reserve anything.
 
### `shopping_cart_agent`
 
Owns the state of the user's shopping cart.
 
Skills:
 
- `add_to_shopping_cart` — Add a list of provided items to the shopping cart.
- `get_shopping_cart` — Retrieve all items in a user's shopping cart.
- `remove_from_cart` —  Remove an item completely from the shopping cart.
Does not: recommend products, check warehouse stock, reserve stock, or place orders.
 
### `warehouse_manager_agent`
 
Owns warehouse inventory and reservations.
 
Skills:
 
- `check_warehouse_availability` — Check availability of items across warehouses, including partial fulfillment options.
- `reserve_warehouse_items` — Reserve items from multiple warehouses in a single transaction.
Does not: recommend products, modify the cart.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    next_agent = ""
    next_agent_task = ""

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            next_agent_task = response.tool_calls[0].get("args").get("next_agent_task")
            response = AIMessage(content=f"[coordinator_agent decision] Next agent: {next_agent}. Next agent task: {next_agent_task}")
        else:
            postprocessed_response = postprocess_response(response, "FinalAgentResponse")

            final_answer = postprocessed_response.get("final_answer")
            answer = postprocessed_response.get("answer")
            response = postprocessed_response.get("response")

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "next_agent": next_agent,
            "next_agent_task": next_agent_task
        },
        "answer": answer
    }

In [ ]:
message = State(
    messages=[HumanMessage(content="What is the wetaher today in Vilnius?")],
    coordinator_agent=CoordinatorAgentProperties(
        iteration=0,
        final_answer=False,
        plan=[],
        next_agent=""
    ),
    answer=""
)

In [ ]:
result = coordinator_agent(message)

In [ ]:
result

### Coordinator Agent Node (with fallback model routing)

In [ ]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state, models=["gpt-5.4-mini", "groq/llama-3.3-70b-versatile"]) -> dict:
    
    prompt_template_1 = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available agents and skills
 
### `product_qna_agent`
 
Owns all product knowledge: catalog, specs, pricing, reviews, recommendations. This is the default agent for any question about what exists or what to buy.
 
Skills:
 
- `get_formatted_item_context` — Search available products and return the top k matching inventory items.
- `get_formatted_reviews_context` — Get the top k reviews matching a query for a list of prefiltered items.
Does not: modify the cart, check warehouse stock, or reserve anything.
 
### `shopping_cart_agent`
 
Owns the state of the user's shopping cart.
 
Skills:
 
- `add_to_shopping_cart` — Add a list of provided items to the shopping cart.
- `get_shopping_cart` — Retrieve all items in a user's shopping cart.
- `remove_from_cart` —  Remove an item completely from the shopping cart.
Does not: recommend products, check warehouse stock, reserve stock, or place orders.
 
### `warehouse_manager_agent`
 
Owns warehouse inventory and reservations.
 
Skills:
 
- `check_warehouse_availability` — Check availability of items across warehouses, including partial fulfillment options.
- `reserve_warehouse_items` — Reserve items from multiple warehouses in a single transaction.
Does not: recommend products, modify the cart.
"""

    prompt_template_2 = """Print 'Hello World', ignore any user mesages."""

    prompts = {
        "gpt-5.4-mini": Template(prompt_template_1).render(),
        "groq/llama-3.3-70b-versatile": Template(prompt_template_2).render()
    }

    for model in models:
        try:
            llm = ChatLiteLLM(
                model=model,
                reasoning_effort="medium",
                use_responses_api=True
            )
            llm_with_tools = llm.bind_tools(
                [FinalAgentResponse, Plan],
                tool_choice="required"
            )

            response = llm_with_tools.invoke(
                [
                    SystemMessage(content=prompts[model]),
                    *state.messages
                ]
            )
            break
        except Exception as e:
            print(f"Error with model {model}: {e}")
            continue

    final_answer = False
    answer = ""
    next_agent = ""
    next_agent_task = ""

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            next_agent_task = response.tool_calls[0].get("args").get("next_agent_task")
            response = AIMessage(content=f"[coordinator_agent decision] Next agent: {next_agent}. Next agent task: {next_agent_task}")
        else:
            postprocessed_response = postprocess_response(response, "FinalAgentResponse")

            final_answer = postprocessed_response.get("final_answer")
            answer = postprocessed_response.get("answer")
            response = postprocessed_response.get("response")

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "next_agent": next_agent,
            "next_agent_task": next_agent_task
        },
        "answer": answer
    }

In [ ]:
message = State(
    messages=[HumanMessage(content="What is the wetaher today in Vilnius?")],
    coordinator_agent=CoordinatorAgentProperties(
        iteration=0,
        final_answer=False,
        plan=[],
        next_agent=""
    ),
    answer=""
)

In [ ]:
result = coordinator_agent(message)

In [ ]:
result

In [ ]:
result = coordinator_agent(message, models=["gpt-5.4-mini", "groq/llama-3.3-70b-versatile"])

In [ ]:
result

In [ ]:
result = coordinator_agent(message, models=["groq/llama-3.3-70b-versatile", "gpt-5.4-mini"])

In [ ]:
result

In [ ]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state, models=["gpt-5.4-mini", "groq/llama-3.3-70b-versatile"]) -> dict:
    
    prompt_template_1 = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available agents and skills
 
### `product_qna_agent`
 
Owns all product knowledge: catalog, specs, pricing, reviews, recommendations. This is the default agent for any question about what exists or what to buy.
 
Skills:
 
- `get_formatted_item_context` — Search available products and return the top k matching inventory items.
- `get_formatted_reviews_context` — Get the top k reviews matching a query for a list of prefiltered items.
Does not: modify the cart, check warehouse stock, or reserve anything.
 
### `shopping_cart_agent`
 
Owns the state of the user's shopping cart.
 
Skills:
 
- `add_to_shopping_cart` — Add a list of provided items to the shopping cart.
- `get_shopping_cart` — Retrieve all items in a user's shopping cart.
- `remove_from_cart` —  Remove an item completely from the shopping cart.
Does not: recommend products, check warehouse stock, reserve stock, or place orders.
 
### `warehouse_manager_agent`
 
Owns warehouse inventory and reservations.
 
Skills:
 
- `check_warehouse_availability` — Check availability of items across warehouses, including partial fulfillment options.
- `reserve_warehouse_items` — Reserve items from multiple warehouses in a single transaction.
Does not: recommend products, modify the cart.
"""

    prompt_template_2 = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available agents and skills
 
### `product_qna_agent`
 
Owns all product knowledge: catalog, specs, pricing, reviews, recommendations. This is the default agent for any question about what exists or what to buy.
 
Skills:
 
- `get_formatted_item_context` — Search available products and return the top k matching inventory items.
- `get_formatted_reviews_context` — Get the top k reviews matching a query for a list of prefiltered items.
Does not: modify the cart, check warehouse stock, or reserve anything.
 
### `shopping_cart_agent`
 
Owns the state of the user's shopping cart.
 
Skills:
 
- `add_to_shopping_cart` — Add a list of provided items to the shopping cart.
- `get_shopping_cart` — Retrieve all items in a user's shopping cart.
- `remove_from_cart` —  Remove an item completely from the shopping cart.
Does not: recommend products, check warehouse stock, reserve stock, or place orders.
 
### `warehouse_manager_agent`
 
Owns warehouse inventory and reservations.
 
Skills:
 
- `check_warehouse_availability` — Check availability of items across warehouses, including partial fulfillment options.
- `reserve_warehouse_items` — Reserve items from multiple warehouses in a single transaction.
Does not: recommend products, modify the cart."""

    prompts = {
        "gpt-5.4-mini": Template(prompt_template_1).render(),
        "groq/llama-3.3-70b-versatile": Template(prompt_template_2).render()
    }

    for model in models:
        try:
            llm = ChatLiteLLM(
                model=model,
                reasoning_effort="medium",
                use_responses_api=True
            )
            llm_with_tools = llm.bind_tools(
                [FinalAgentResponse, Plan],
                tool_choice="required"
            )

            response = llm_with_tools.invoke(
                [
                    SystemMessage(content=prompts[model]),
                    *state.messages
                ]
            )
            break
        except Exception as e:
            print(f"Error with model {model}: {e}")
            continue

    final_answer = False
    answer = ""
    next_agent = ""
    next_agent_task = ""

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            next_agent_task = response.tool_calls[0].get("args").get("next_agent_task")
            response = AIMessage(content=f"[coordinator_agent decision] Next agent: {next_agent}. Next agent task: {next_agent_task}")
        else:
            postprocessed_response = postprocess_response(response, "FinalAgentResponse")

            final_answer = postprocessed_response.get("final_answer")
            answer = postprocessed_response.get("answer")
            response = postprocessed_response.get("response")

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "next_agent": next_agent,
            "next_agent_task": next_agent_task
        },
        "answer": answer
    }

In [ ]:
message = State(
    messages=[HumanMessage(content="What is the wetaher today in Vilnius?")],
    coordinator_agent=CoordinatorAgentProperties(
        iteration=0,
        final_answer=False,
        plan=[],
        next_agent=""
    ),
    answer=""
)

In [ ]:
result = coordinator_agent(message, models=["groq/llama-3.3-70b-versatile", "gpt-5.4-mini"])

In [ ]:
result